# PARTE 2: Classificação de Gestos (Língua Gestual)

**Licenciatura em Engenharia de Sistemas e Tecnologias Informáticas** 

**Unidade Curricular:** Aprendizagem Automática  
**Ano Letivo:** 2025/2026 

---

### Identificação do Grupo
* **Aluno 1:** Bernardo Freitas (79295)
* **Docente:** Prof. Pedro Cardoso

---

Nesta segunda fase do projeto, o objetivo muda de Regressão (prever um valor contínuo) para **Classificação Multiclasse**. O objetivo é treinar um modelo capaz de receber coordenadas da mão (extraídas via MediaPipe) e identificar qual a letra do alfabeto (A-Z) correspondente.

### Metodologia de Treino
Testei 5 famílias de algoritmos para encontrar o melhor compromisso entre **Acurácia** (precisão) e **Tempo de Inferência** (velocidade para uso em tempo real):

1.  **KNN (K-Nearest Neighbors):** Baseado em distância.
2.  **Random Forest:** Ensemble de árvores de decisão.
3.  **Decision Tree:** Árvore simples (baseline).
4.  **SVM (Support Vector Machines):** Procura a melhor margem de separação.
5.  **MLP (Multi-Layer Perceptron):** Rede Neuronal simples.

### Pré-processamento Crítico
Diferente das árvores (XGBoost/Random Forest), modelos baseados em distância (KNN e SVM) e baseados em gradiente (MLP) exigem que os dados estejam na mesma escala. Por isso, apliquei o **StandardScaler** ($\mu=0, \sigma=1$) em todas as coordenadas.

# 1. Configuração e Importação de Bibliotecas

Importação das bibliotecas necessárias para manipulação de dados, treino de modelos de classificação e serialização (salvamento) dos artefactos finais.

In [1]:
import pandas as pd
import numpy as np
import time
import pickle
import warnings

# Sklearn - Processamento e Seleção
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Sklearn - Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Ignorar warnings de convergência (comuns em MLP se treinados rápido)
warnings.filterwarnings('ignore')

# 2. Pré-processamento e Divisão Estratificada

Nesta etapa, carrego o dataset de landmarks. Diferente da regressão (preços), aqui aplico duas técnicas cruciais para classificação:

1.  **Divisão Estratificada (`stratify`):** Garante que todas as letras do alfabeto (A-Z) estão representadas proporcionalmente nos conjuntos de treino e teste.
2.  **StandardScaler:** Normaliza as coordenadas para média 0 e desvio padrão 1. Isto é **obrigatório** para o bom funcionamento do SVM e KNN.

In [ ]:
def load_and_preprocess_gestures(csv_path="landmarks_dataset.csv"):
    """
    Carrega, codifica e normaliza os dados de gestos.
    Retorna os conjuntos divididos e os objetos de transformação (scaler, encoder).
    """
    print("A carregar dataset...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"ERRO: Ficheiro '{csv_path}' não encontrado.")
        return None

    # 1. Codificar a Mão (Left/Right -> 0/1)
    le_hand = LabelEncoder()
    df['hand'] = le_hand.fit_transform(df['hand'])

    # Separar Features (X) e Target (y)
    X = df.drop(['label'], axis=1) 
    y = df['label']

    print(f"Dados Carregados: {X.shape[0]} amostras | {X.shape[1]} features")

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp
    )

    # 2. Normalização
    scaler = StandardScaler()

    # Fit apenas no treino para evitar data leakage
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    data_bundle = {
        'X_train': X_train_scaled, 'y_train': y_train,
        'X_val': X_val_scaled, 'y_val': y_val,
        'X_test': X_test_scaled, 'y_test': y_test
    }

    return data_bundle, scaler, le_hand

# 3. Configuração do Espaço de Busca

Definimos os 5 algoritmos a testar e os respetivos hiperparâmetros para o `GridSearch`. O objetivo é testar famílias diferentes de classificadores.

In [ ]:
def get_model_configs():
    """Retorna o dicionário com modelos e hiperparâmetros a testar."""
    return {
        'KNN': {
            'model': KNeighborsClassifier(),
            'params': {
                'n_neighbors': [3, 5, 7],
                'weights': ['distance'] # 'distance' dá mais peso a vizinhos próximos
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [100], 
                'max_depth': [None, 20] # Controla overfitting
            }
        },
        'DecisionTree': { # Baseline simples
            'model': DecisionTreeClassifier(random_state=42),
            'params': { 'max_depth': [None, 15] }
        },
        'SVM': {
            'model': SVC(random_state=42),
            'params': {
                'C': [1, 10], 
                'kernel': ['rbf'] # Kernel não-linear para gestos complexos
            }
        },
        'MLP (Neural Net)': {
            'model': MLPClassifier(random_state=42, max_iter=300),
            'params': {
                'hidden_layer_sizes': [(64, 64), (128,128)], # Duas camadas ocultas (neurônios)
                'activation': ['relu'] # transforma todos os valores negativos em zero e mantém os positivos.
            }
        }
    }

# 4. Loop de Treino e Seleção de Modelo

Esta função itera sobre todos os modelos definidos, executa a otimização de hiperparâmetros (GridSearch) e avalia dois critérios:
1.  **Acurácia:** Capacidade de acertar na letra.
2.  **Tempo de Inferência:** Velocidade de previsão (crítico para uso em webcam).

In [ ]:
def train_and_select_best(data, models_config):
    
    results = []
    best_model_obj = None
    best_score = 0
    best_time = float('inf')

    print(f"{'Modelo':<20} | {'Acurácia':<10} | {'Tempo (ms)':<10}")
    print("-" * 45)

    for name, config in models_config.items():
        
        # 1. Grid Search (Treino)
        grid = GridSearchCV(config['model'], config['params'], cv=3, n_jobs=-1)
        grid.fit(data['X_train'], data['y_train'])
        
        # 2. Avaliação na Validação
        best_clf = grid.best_estimator_
        val_preds = best_clf.predict(data['X_val'])
        accuracy = accuracy_score(data['y_val'], val_preds)
        
        # 3. Teste de Stress de Velocidade (Tempo de Inferência)
        # Simulo 100 previsões unitárias para ver se aguenta a webcam
        start = time.time()
        for _ in range(100):
            best_clf.predict(data['X_val'][:1])
        avg_time_ms = ((time.time() - start) / 100) * 1000 # Converter para milissegundos

        print(f"{name:<20} | {accuracy:.4f}     | {avg_time_ms:.4f} ms")

        # Guardar métricas
        results.append({
            'Modelo': name,
            'Accuracy': accuracy,
            'Inference_Time_ms': avg_time_ms,
            'Best_Params': grid.best_params_
        })

        # 4. Lógica de Seleção (Prioridade: Acurácia > Velocidade)
        if accuracy > best_score:
            best_score = accuracy
            best_time = avg_time_ms
            best_model_obj = best_clf
        elif accuracy == best_score and avg_time_ms < best_time:
            # Desempate pela velocidade
            best_time = avg_time_ms
            best_model_obj = best_clf

    return best_model_obj, pd.DataFrame(results)

# 5. Execução e Persistência

Executo o pipeline completo. O modelo vencedor é avaliado no conjunto de teste final (dados nunca vistos) e exportado juntamente com o `scaler` e o `encoder`.

**Artefactos gerados:**
* `best_model_asl.pkl`: O cérebro (modelo treinado).
* `scaler.pkl`: A régua (normalizador de dados).
* `hand_encoder.pkl`: O tradutor (Esquerda/Direita).

In [5]:
if __name__ == "__main__":
    
    # 1. Carregar Dados
    dataset = load_and_preprocess_gestures()
    
    if dataset:
        data_bundle, scaler_obj, hand_encoder_obj = dataset
        
        # 2. Obter Configurações
        model_zoo = get_model_configs()
        
        # 3. Treinar e Escolher Vencedor
        print("\nINICIA OTIMIZAÇÃO...")
        best_model, df_results = train_and_select_best(data_bundle, model_zoo)
        
        # 4. Análise do Vencedor
        print(f"\nVENCEDOR: {best_model.__class__.__name__}")
        
        # Avaliação Final (Conjunto de Teste)
        print("\nRelatório Final (Test Set):")
        test_preds = best_model.predict(data_bundle['X_test'])
        print(classification_report(data_bundle['y_test'], test_preds))
        
        # 5. Guardar Ficheiros (Essencial para a Webcam)
        print("A guardar ficheiros .pkl...")
        
        with open('best_model_asl.pkl', 'wb') as f:
            pickle.dump(best_model, f)
            
        with open('scaler.pkl', 'wb') as f:
            pickle.dump(scaler_obj, f)
            
        with open('hand_encoder.pkl', 'wb') as f:
            pickle.dump(hand_encoder_obj, f)
            
        # Guardar CSV de comparação para o relatório
        df_results.to_csv('resultados_classificacao.csv', index=False)

A carregar dataset...
Dados Carregados: 25566 amostras | 64 features

INICIA OTIMIZAÇÃO...
Modelo               | Acurácia   | Tempo (ms)
---------------------------------------------
KNN                  | 1.0000     | 1.0009 ms
RandomForest         | 1.0000     | 2.4848 ms
DecisionTree         | 0.9940     | 0.1927 ms
SVM                  | 1.0000     | 0.3549 ms
MLP (Neural Net)     | 1.0000     | 0.1303 ms

VENCEDOR: MLPClassifier

Relatório Final (Test Set):
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       150
           B       1.00      1.00      1.00       150
           C       1.00      1.00      1.00       142
           D       1.00      1.00      1.00       150
           E       1.00      1.00      1.00       150
           F       1.00      1.00      1.00       150
           G       1.00      1.00      1.00       149
           H       1.00      1.00      1.00       150
           I       1.00      1.00      1.00  